# MOOC 03 — Continuer la recherche

But : transformer l'état du dépôt en plan d'action expérimental défendable.

Ce notebook ne lance pas de gros jobs. Il génère des commandes, des checklists et des critères go/no-go.

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
print(ROOT)

## 1. Thèse de recherche en une phrase

L'échantillonnage uniforme optimise surtout la moyenne sur l'attracteur. Le projet teste si un générateur d'états difficiles, mélangé à une fraction uniforme `alpha`, améliore la couverture des régions rares/off-attractor et donc la robustesse rollout.

Conséquence méthodologique : le bulk uniforme n'est pas la métrique principale. Les métriques principales sont hard/tube/rollout, avec baselines budget-matched.

## 2. Scorecard go/no-go

Pour accepter une variante générative, exiger :

- **Budget** : même nombre d'appels solveur utiles que les baselines ;
- **Hard/tube** : amélioration nette sur `val_hard_tv_div` et `val_tube_*` ;
- **Baselines** : mieux que `random_tube` et `mined_ic`, sinon la génération apprise n'est pas justifiée ;
- **Bulk** : pas d'effondrement massif sur `val/nrmse_mean` ;
- **Réalisme** : TV/spectre/amplitude plausibles, pas de NaN solveur ;
- **Audit** : `target_bins`, `pretrain_uncertainty`, `source` persistés et inspectables ;
- **Négatifs** : lowamp-hard et corrélation de conditionnement doivent être rapportés honnêtement.

In [ ]:
scorecard = {
    "budget_matched": "rounds and transitions matched across arms",
    "beats_uniform_on_hard_tube": "hard_val and tube metrics lower than uniform",
    "beats_random_tube": "generator justified only if better than no-learning tube baseline",
    "bulk_guard": "no large degradation on val/nrmse_mean",
    "realism": "TV/PSD/amplitude close to validation states",
    "auditability": "source, target_bins, pretrain losses/uncertainty persisted",
    "negative_results": "lowamp-hard and failed conditioning retained",
}
for k, v in scorecard.items():
    print(f"{k:32s} {v}")

## 3. Prochaines expériences prioritaires

D'après `HANDOFF.md`, l'ordre raisonnable est :

1. Lire les résultats du pilot v3 ;
2. Si `gen_v3*` bat `random_tube` sur hard/tube diverse suite, lancer 5 seeds des bras gagnants ;
3. Mesurer la fidélité de conditionnement avec `target_bins` vs désaccord réalisé ;
4. Regénérer Burgers avec stability gate avant toute claim Burgers ;
5. Traiter lowamp-hard avec un second head ou une métrique de difficulté amplitude-aware ;
6. Réécrire le papier depuis les piliers coverage/DRO, alpha-mixture, CVaR/tail sampling et rollout tube.

In [ ]:
sections = ["## Immediate next steps", "## Known open issues"]
lines = (ROOT / "HANDOFF.md").read_text().splitlines()
for section in sections:
    print(f"\n{section}")
    try:
        start = next(i for i, line in enumerate(lines) if line.startswith(section))
    except StopIteration:
        continue
    end = next((i for i in range(start + 1, len(lines)) if lines[i].startswith("## ")), len(lines))
    print("\n".join(lines[start + 1:end]))

## 4. Générateur de commandes locales

Ces commandes sont petites et utiles pour vérifier l'environnement avant de lancer des campagnes.

In [ ]:
local_commands = {
    "smoke conditional": ".venv/bin/python -m poolbased_surrogate.run configs/smoke.yaml --fresh",
    "smoke v3 cpu": ".venv/bin/python -m poolbased_surrogate.run configs/smoke_v3.yaml --fresh",
    "validation only": ".venv/bin/python -m poolbased_surrogate.run configs/smoke.yaml --create-validation-only",
}
for name, cmd in local_commands.items():
    print("# " + name)
    print(cmd)
    print()


## 5. Générateur de commandes hard/diverse validation

À utiliser quand tu as une banque uniforme stable. Ne réutilise pas une banque contaminée par des artefacts solveur.

In [ ]:
def build_diverse_command(bank, config, out_dir):
    return f""".venv/bin/python scripts/build_diverse_validation.py \
  --bank {bank} \
  --config {config} \
  --output-dir {out_dir} \
  --seed 0"""

print(build_diverse_command(
    bank="<uniform_validation_bank.npz>",
    config="configs/bigfoot_ks_v3_base.yaml",
    out_dir="<validation_diverse_suite>",
))

## 6. Générateur de commandes post-hoc

Le post-hoc reconstruit le surrogate depuis `config.resolved.json` et `checkpoint_latest.pt`. Il évite de relancer l'entraînement pour tester de nouvelles métriques.

In [ ]:
def hard_posthoc(runs, uniform, hard, baseline="uniform_baseline"):
    runs_str = " ".join(runs)
    return f""".venv/bin/python scripts/eval_hard_posthoc.py \
  --runs {runs_str} \
  --uniform {uniform} \
  --hard {hard} \
  --baseline {baseline}"""

def rollout_posthoc(runs, uniform, baseline="uniform_baseline", steps=100):
    runs_str = " ".join(runs)
    return f""".venv/bin/python scripts/eval_rollout_posthoc.py \
  --runs {runs_str} \
  --uniform {uniform} \
  --steps {steps} \
  --baseline {baseline}"""

example_runs = ["<campaign>/uniform_baseline_seed101", "<campaign>/random_tube_seed101", "<campaign>/gen_v3_edit_seed101"]
print(hard_posthoc(example_runs, "<uniform.npz>", "<hard_dir>"))
print()
print(rollout_posthoc(example_runs, "<uniform.npz>"))

## 7. Bras expérimentaux minimaux

Pour une campagne propre, ne pars pas directement sur trop de variantes. Un set minimal défendable :

- `uniform_baseline` : contrôle ;
- `noise_inject` : baseline robustesse zéro-génération ;
- `random_tube` : tube aléatoire sans apprentissage ;
- `mined_ic` ou `tube_select` : sélection sans génération ;
- `gen_v3` : génération apprise scratch ;
- `gen_v3_edit` : génération apprise SDEdit autour d'ancres.

In [ ]:
arms = ["uniform_baseline", "noise_inject", "random_tube", "tube_select", "mined_ic", "gen_v3", "gen_v3_edit"]
seeds = [101, 202, 303, 404, 505]
print("arms", arms)
print("seeds", seeds)
print("total jobs", len(arms) * len(seeds))

## 8. Critères de rédaction

Ce qui peut devenir une claim papier :

- une amélioration hard/tube robuste multi-seed ;
- une baseline `random_tube` battue ;
- une explication théorique coverage/DRO compatible avec les résultats ;
- une analyse d'échec lowamp-hard et bulk non-neutre ;
- une généralisation Burgers seulement après banque stable.

Ce qui ne suffit pas :

- un seul run smoke ;
- un gain bulk moyen ;
- un générateur qui produit juste des états rugueux ;
- un rollout amélioré sur une seed mais instable sur les autres.

## 9. Plan personnel de reprise en 2 jours

**Jour 1**

- Lire `HANDOFF.md`, `docs/roadmap.md`, puis ce MOOC ;
- exécuter les notebooks 00–02 ;
- relancer un smoke si nécessaire ;
- identifier les campagnes disponibles localement ou sur scratch.

**Jour 2**

- lancer `eval_hard_posthoc.py` et `eval_rollout_posthoc.py` sur les meilleurs runs disponibles ;
- remplir la scorecard ;
- décider go/no-go pour v3 ;
- si go : planifier 5 seeds ; sinon : isoler réalisme, lowamp ou steering comme prochain bug.